# 01 – PyIceberg

> Runs against the catalog from `CATALOG` (Lakekeeper, Polaris, RustFS or Unity; default Lakekeeper). What a catalog cannot do is shown by the cell instead of aborting.

PyIceberg is the Python reference implementation. It talks to the REST catalog directly and
supports **both** access modes (remote signing and vended credentials).

> Limitation (PyIceberg 0.12): **cannot write v3 tables** and cannot read **position deletes from Trino**.

> This notebook uses the warehouse with vended credentials (Lakekeeper: `demo-sts`). With remote signing (Lakekeeper only, `demo-signing`) about 1–10 % of PyIceberg commits fail intermittently with 403 – a race condition in PyIceberg/s3fs, see README.

In [ ]:
from iceberg_lab.config import PROFILE, REMOTE_SIGNING, SETTINGS, WAREHOUSE_SIGNING, WAREHOUSE_STS, WAREHOUSES
from iceberg_lab.notebook import BUCKET, TRINO_SIGNING, TRINO_STS, catalog_limit, drop_table, header

CATALOG_URI = SETTINGS.catalog_uri
S3_ENDPOINT = SETTINGS.s3_endpoint
header()

In [ ]:
from iceberg_lab.engines import pyiceberg_catalog

catalog = pyiceberg_catalog(WAREHOUSE_STS)
NS = "nb_pyiceberg"
catalog.create_namespace_if_not_exists(NS)
for ident in catalog.list_tables(NS):
    drop_table(catalog, ident)
catalog.list_namespaces()

## Create a partitioned table

In [ ]:
from pyiceberg.partitioning import PartitionField, PartitionSpec
from pyiceberg.schema import Schema
from pyiceberg.transforms import DayTransform
from pyiceberg.types import DoubleType, LongType, NestedField, StringType, TimestampType

schema = Schema(
    NestedField(1, "id", LongType(), required=False),
    NestedField(2, "ts", TimestampType(), required=False),
    NestedField(3, "city", StringType(), required=False),
    NestedField(4, "amount", DoubleType(), required=False),
)
spec = PartitionSpec(PartitionField(source_id=2, field_id=1000, transform=DayTransform(), name="ts_day"))
orders = catalog.create_table(f"{NS}.orders", schema=schema, partition_spec=spec)
orders

## Daten schreiben (append)

In [ ]:
from datetime import datetime

import pyarrow as pa

batch = pa.Table.from_pylist([
    {"id": 1, "ts": datetime(2026, 11, 1, 9), "city": "Berlin", "amount": 12.5},
    {"id": 2, "ts": datetime(2026, 11, 1, 17), "city": "Hamburg", "amount": 99.0},
    {"id": 3, "ts": datetime(2026, 11, 2, 8), "city": "Berlin", "amount": 7.25},
], schema=orders.schema().as_arrow())
orders.append(batch)
orders.scan().to_pandas()

## Read with filter and projection – partition pruning

In [ ]:
from pyiceberg.expressions import And, EqualTo, GreaterThanOrEqual

scan = orders.scan(
    row_filter=And(EqualTo("city", "Berlin"), GreaterThanOrEqual("ts", "2026-11-02T00:00:00")),
    selected_fields=("id", "amount"),
)
print("Files after pruning:", len(list(scan.plan_files())), "of", len(list(orders.scan().plan_files())))
scan.to_arrow()

## Upsert and delete

In [ ]:
changes = pa.Table.from_pylist([
    {"id": 2, "ts": datetime(2026, 11, 1, 17), "city": "Hamburg", "amount": 120.0},
    {"id": 4, "ts": datetime(2026, 11, 3, 12), "city": "Munich", "amount": 5.0},
], schema=orders.schema().as_arrow())
res = orders.upsert(changes, join_cols=["id"])
print("updated:", res.rows_updated, "inserted:", res.rows_inserted)

orders.delete(EqualTo("id", 1))
orders.scan().to_pandas().sort_values("id")

## Schema evolution

> **Known PyIceberg 0.12 bug:** after `rename_column`, `upsert` fails because PyIceberg
> compares the existing rows with the old column names from the Parquet files
> (see `tests/specific/test_pyiceberg.py::test_upsert_after_rename`). `append`, `overwrite`, `delete` and reading work.

In [ ]:
with orders.update_schema() as u:
    u.add_column("channel", StringType(), doc="web | store")
    u.rename_column("amount", "total")
orders = orders.refresh()

orders.append(pa.Table.from_pylist(
    [{"id": 5, "ts": datetime(2026, 11, 4, 10), "city": "Cologne", "total": 33.0, "channel": "web"}],
    schema=orders.schema().as_arrow()))
orders.scan().to_pandas().sort_values("id")

## Time travel, snapshots, branches and tags

In [ ]:
orders.inspect.snapshots().to_pandas()[["committed_at", "snapshot_id", "operation"]]

In [ ]:
first = orders.history()[0].snapshot_id
print("Rows in the first snapshot:", orders.scan(snapshot_id=first).to_arrow().num_rows)

orders.manage_snapshots().create_tag(first, "initial-load").create_branch(first, "experiment").commit()
orders.refresh().metadata.refs

## Metadata tables

In [ ]:
orders.inspect.files().to_pandas()[["file_path", "record_count", "file_size_in_bytes"]]

In [ ]:
orders.inspect.partitions().to_pandas()

## DataFrame-Ausgaben: pandas, Arrow, DuckDB

In [ ]:
con = orders.scan().to_duckdb(table_name="orders")
con.sql("SELECT city, sum(total) FROM orders GROUP BY city ORDER BY 2 DESC").df()